Recommended

In [ ]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'

Connect To Drive And Import Libraries

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
import json, time
import numpy as np
import pandas as pd
import requests
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import userdata

PROJECT_ROOT = '/content/drive/MyDrive/Projects/multi-agent-discovery'
DATA     = os.path.join(PROJECT_ROOT, 'data/raw/ml-32m')
TMDB_DIR = os.path.join(PROJECT_ROOT, 'data/raw/tmdb')
os.makedirs(TMDB_DIR, exist_ok=True)
OUT_PATH = os.path.join(TMDB_DIR, 'overviews.jsonl')

TMDB_API_KEY = userdata.get('TMDB_API_KEY')
print("out path:", OUT_PATH)

Mounted at /content/drive
out path: /content/drive/MyDrive/Projects/multi-agent-discovery/data/raw/tmdb/overviews.jsonl


build the list of fetch targets

In [ ]:
links = pd.read_csv(os.path.join(DATA, 'links.csv'))
targets = list(links.dropna(subset=['tmdbId'])[['movieId', 'tmdbId']]
               .itertuples(index=False, name=None))     # [(movieId, tmdbId), ...]
print(f"fetch targets (movies with a tmdbId): {len(targets):,}")

fetch targets (movies with a tmdbId): 87,461


the field extractor (same as Day 2, redefined here) + single-movie fetch with retry

In [ ]:
def extract_fields(m):
    crew = m.get('credits', {}).get('crew', [])
    cast = m.get('credits', {}).get('cast', [])
    director = next((c['name'] for c in crew if c.get('job') == 'Director'), None)
    rd = m.get('release_date') or ''
    year = int(rd[:4]) if rd[:4].isdigit() else None
    return {
        'tmdb_id':  m.get('id'),
        'title':    m.get('title'),
        'overview': m.get('overview') or '',
        'genres':   [g['name'] for g in m.get('genres', [])],
        'runtime':  m.get('runtime'),
        'year':     year,
        'director': director,
        'cast':     [c['name'] for c in cast[:5]],
    }

session = requests.Session()   # reuse connections -> faster

def fetch_one(movie_id, tmdb_id, max_retries=4):
    url = f"https://api.themoviedb.org/3/movie/{int(tmdb_id)}"
    params = {"api_key": TMDB_API_KEY, "append_to_response": "credits"}
    for attempt in range(max_retries):
        try:
            r = session.get(url, params=params, timeout=15)
            if r.status_code == 200:
                rec = extract_fields(r.json())
                rec['movieId'], rec['status'] = int(movie_id), 'ok'
                return rec
            if r.status_code == 404:                       # dead link -> done, don't retry
                return {'movieId': int(movie_id), 'tmdb_id': int(tmdb_id), 'status': 'not_found'}
            if r.status_code == 429:                        # rate limited -> wait and retry
                time.sleep(int(r.headers.get('Retry-After', 2 ** attempt)))
                continue
            time.sleep(2 ** attempt)                        # 5xx etc -> backoff
        except requests.RequestException:
            time.sleep(2 ** attempt)                        # network error -> backoff
    return {'movieId': int(movie_id), 'tmdb_id': int(tmdb_id), 'status': 'failed'}

the resumable driver

In [ ]:
def load_done(path):
    done = set()
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                try:
                    done.add(json.loads(line)['movieId'])
                except Exception:
                    pass          # skip a torn last line from a crash
    return done

def run_fetch(targets, out_path, max_workers=8, flush_every=300):
    done = load_done(out_path)
    todo = [(m, t) for (m, t) in targets if m not in done]
    print(f"already done: {len(done):,} | to fetch this run: {len(todo):,}")
    buffer, n_ok, n_bad = [], 0, 0
    with open(out_path, 'a') as fout, ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(fetch_one, m, t): m for (m, t) in todo}
        for i, fut in enumerate(as_completed(futures), 1):
            rec = fut.result()
            buffer.append(rec)
            n_ok += (rec.get('status') == 'ok')
            n_bad += (rec.get('status') != 'ok')
            if len(buffer) >= flush_every:
                fout.write('\n'.join(json.dumps(r) for r in buffer) + '\n'); fout.flush()
                buffer.clear()
                print(f"  {i:,}/{len(todo):,} | ok={n_ok:,} bad={n_bad:,}")
        if buffer:
            fout.write('\n'.join(json.dumps(r) for r in buffer) + '\n'); fout.flush()
    print(f"run finished. ok={n_ok:,} bad={n_bad:,}")

validate on a tiny sample first

In [ ]:
SAMPLE = os.path.join(TMDB_DIR, '_sample.jsonl')
if os.path.exists(SAMPLE): os.remove(SAMPLE)

run_fetch(targets[:20], SAMPLE, max_workers=4, flush_every=5)
print("\n--- records written ---")
with open(SAMPLE) as f:
    for line in list(f)[:3]:
        print(json.loads(line))

print("\n--- resume test (should say 'to fetch this run: 0') ---")
run_fetch(targets[:20], SAMPLE, max_workers=4)
os.remove(SAMPLE)   # clean up the sample

already done: 0 | to fetch this run: 20
  5/20 | ok=5 bad=0
  10/20 | ok=10 bad=0
  15/20 | ok=15 bad=0
  20/20 | ok=20 bad=0
run finished. ok=20 bad=0

--- records written ---
{'tmdb_id': 862, 'title': 'Toy Story', 'overview': "Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.", 'genres': ['Family', 'Comedy', 'Animation', 'Adventure'], 'runtime': 81, 'year': 1995, 'director': 'John Lasseter', 'cast': ['Tom Hanks', 'Tim Allen', 'Don Rickles', 'Jim Varney', 'Wallace Shawn'], 'movieId': 1, 'status': 'ok'}
{'tmdb_id': 15602, 'title': 'Grumpier Old Men', 'overview': "A family wedding reignites the ancient feud between next-door neighbors and fishing buddies John and Max. Meanwhile, a sultry Italian divorcée opens a restaurant at the local b

the real fetch

In [ ]:
run_fetch(targets, OUT_PATH, max_workers=8, flush_every=300)

already done: 87,461 | to fetch this run: 0
run finished. ok=0 bad=0


progress / status check

In [ ]:
done = load_done(OUT_PATH)
print(f"fetched: {len(done):,} / {len(targets):,}  ({len(done)/len(targets)*100:.1f}%)")

statuses = Counter()
with open(OUT_PATH) as f:
    for line in f:
        try: statuses[json.loads(line)['status']] += 1
        except: pass
print("status breakdown:", dict(statuses))

fetched: 87,461 / 87,461  (100.0%)
status breakdown: {'ok': 86262, 'not_found': 1199}


Install Transformers

In [ ]:
!pip install -q faiss-cpu "sentence-transformers>=3.0.0" "transformers>=4.51.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 26.9 MB/s eta 0:00:00


Connect To Drive and install Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
import torch

TMDB_DIR    = os.path.join(PROJECT_ROOT, 'data/raw/tmdb')
CATALOG_DIR = os.path.join(PROJECT_ROOT, 'data/processed/catalog')
FAISS_DIR   = os.path.join(PROJECT_ROOT, 'artifacts/faiss')
os.makedirs(CATALOG_DIR, exist_ok=True); os.makedirs(FAISS_DIR, exist_ok=True)
print("GPU available:", torch.cuda.is_available())     # should be True

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU available: False


load the ok records into the master catalog and save it

In [ ]:
rows = []
with open(os.path.join(TMDB_DIR, 'overviews.jsonl')) as f:
    for line in f:
        try: r = json.loads(line)
        except: continue
        if r.get('status') == 'ok':
            rows.append(r)
catalog = pd.DataFrame(rows)
print("ok records:", len(catalog))

# Master catalog = ALL ok records (feeds Tools C/D; they need metadata even for thin-overview movies)
catalog.to_parquet(os.path.join(CATALOG_DIR, 'catalog.parquet'), index=False)
print(catalog[['movieId','title','runtime','year']].head(3))

ok records: 86262
   movieId           title  runtime    year
0       73  Les Miserables      175  1995.0
1       16          Casino      179  1995.0
2       23       Assassins      132  1995.0


overview length distribution

In [ ]:
catalog['ov_words'] = catalog['overview'].fillna('').str.split().str.len()
print("empty overviews:", int((catalog['ov_words'] == 0).sum()))
print(catalog['ov_words'].describe(percentiles=[.01,.05,.1,.25,.5]).round(1), "\n")
for thr in [0, 3, 5, 10, 15, 20]:
    n = int((catalog['ov_words'] >= thr).sum())
    print(f">= {thr:>2} words: {n:,} ({n/len(catalog)*100:.1f}%)")

empty overviews: 242
count    86262.0
mean        48.6
std         30.3
min          0.0
1%           9.0
5%          15.0
10%         18.0
25%         26.0
50%         40.0
max        190.0
Name: ov_words, dtype: float64 

>=  0 words: 86,262 (100.0%)
>=  3 words: 86,013 (99.7%)
>=  5 words: 85,979 (99.7%)
>= 10 words: 85,315 (98.9%)
>= 15 words: 82,237 (95.3%)
>= 20 words: 75,936 (88.0%)


apply the filter → the set to embed

In [ ]:
MIN_WORDS = 10                                    # drop empty + fragment stubs
embed_df = catalog[catalog['ov_words'] >= MIN_WORDS].reset_index(drop=True)
dropped = len(catalog) - len(embed_df)
print(f"to embed: {len(embed_df):,} | dropped (thin/empty): {dropped:,} ({dropped/len(catalog)*100:.1f}%)")

to embed: 85,315 | dropped (thin/empty): 947 (1.1%)


load Qwen3 and embed the overviews

In [ ]:
from sentence_transformers import SentenceTransformer

EMB_DIM = 512
model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    truncate_dim=EMB_DIM,
    tokenizer_kwargs={"padding_side": "left"},   # <-- CRITICAL: Qwen3 pools the last token
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

texts = embed_df['overview'].tolist()
emb = model.encode(texts, batch_size=64, normalize_embeddings=True,
                   show_progress_bar=True, convert_to_numpy=True).astype('float32')
print("embeddings:", emb.shape)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Batches:   0%|          | 0/1334 [00:00<?, ?it/s]

embeddings: (85315, 512)


build and save the FAISS index + aligned id map

In [ ]:
import faiss
index = faiss.IndexFlatIP(EMB_DIM)   # exact cosine search on normalized vectors
index.add(emb)
print("indexed vectors:", index.ntotal)

faiss.write_index(index, os.path.join(FAISS_DIR, 'plots.index'))
np.save(os.path.join(FAISS_DIR, 'faiss_movieids.npy'), embed_df['movieId'].to_numpy())
print("saved:", os.listdir(FAISS_DIR))

indexed vectors: 85315
saved: ['plots.index', 'faiss_movieids.npy']


semantic_search and test

In [ ]:
QUERY_PROMPT = ("Instruct: Given a movie search query describing plot, mood, or themes, "
                "retrieve movies whose plot matches.\nQuery:")
movieids_arr = embed_df['movieId'].to_numpy()
title_of = catalog.set_index('movieId')['title']

def semantic_search(query_text, n=10):
    q = model.encode([query_text], prompt=QUERY_PROMPT,
                     normalize_embeddings=True, convert_to_numpy=True).astype('float32')
    scores, idx = index.search(q, n)
    return [(int(movieids_arr[j]), float(s)) for j, s in zip(idx[0], scores[0])]

for query in ["mind-bending sci-fi about dreams and reality",
              "heartwarming animated movie about friendship",
              "gritty crime drama with corrupt cops"]:
    print("\nQUERY:", query)
    for mid, sc in semantic_search(query, 5):
        print(f"  {sc:.3f}  {title_of.get(mid, mid)}")


QUERY: mind-bending sci-fi about dreams and reality
  0.574  Dün Gece Bir Rüya Gördüm
  0.573  The Mare
  0.565  unknown
  0.560  Enakkul Oruvan
  0.558  Lucia

QUERY: heartwarming animated movie about friendship
  0.601  Cats & Dogs
  0.598  The Little Polar Bear
  0.576  Wubbzy's Big Movie!
  0.565  Soccer Dog: The Movie
  0.564  Trail of the Panda

QUERY: gritty crime drama with corrupt cops
  0.672  Motta Shiva Ketta Shiva
  0.620  Jo Pil-ho: The Dawning Rage
  0.613  Asura: The City of Madness
  0.610  Raman Raghav 2.0
  0.609  Double Bang


package Tool B

In [2]:
%%writefile /content/drive/MyDrive/Projects/multi-agent-discovery/src/tools/tool_b_semantic.py
"""Tool B — semantic plot search (Qwen3 embeddings + FAISS)."""
import os
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

QUERY_PROMPT = ("Instruct: Given a movie search query describing plot, mood, or themes, "
                "retrieve movies whose plot matches.\nQuery:")


class ToolB:
    def __init__(self, faiss_dir, catalog_path, model_name="Qwen/Qwen3-Embedding-0.6B",
                 emb_dim=512, device=None):
        self.index = faiss.read_index(os.path.join(faiss_dir, "plots.index"))
        self.movieids = np.load(os.path.join(faiss_dir, "faiss_movieids.npy"))
        self.model = SentenceTransformer(model_name, truncate_dim=emb_dim,
                                         processor_kwargs={"padding_side": "left"},
                                         device=device)
        self._title = pd.read_parquet(catalog_path)[["movieId", "title"]].set_index("movieId")["title"]

    def semantic_search(self, query_text, n=50):
        q = self.model.encode([query_text], prompt=QUERY_PROMPT,
                              normalize_embeddings=True, convert_to_numpy=True).astype("float32")
        scores, idx = self.index.search(q, n)
        results = [{"movieId": int(self.movieids[j]), "score": float(s),
                    "title": self._title.get(int(self.movieids[j]), None)}
                   for j, s in zip(idx[0], scores[0])]
        return {"results": results}

Overwriting /content/drive/MyDrive/Projects/multi-agent-discovery/src/tools/tool_b_semantic.py


import the module fresh and smoke-test

In [ ]:
import sys
sys.path.append(os.path.join(PROJECT_ROOT, 'src'))
sys.modules.pop('tools.tool_b_semantic', None)     # evict cache so the rewritten file loads
from tools.tool_b_semantic import ToolB

tool_b = ToolB(faiss_dir=FAISS_DIR,
               catalog_path=os.path.join(CATALOG_DIR, 'catalog.parquet'),
               device='cuda' if torch.cuda.is_available() else 'cpu')
for r in tool_b.semantic_search("space war with rebels against an evil empire", n=5)['results']:
    print(f"  {r['score']:.3f}  {r['title']}")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

  0.575  Wing Commander
  0.570  Occupation: Rainfall
  0.555  Scavengers
  0.554  Escape from Galaxy 3
  0.554  Battle in Space: The Armada Attacks


a popularity signal to the catalog

In [ ]:
CATALOG_PATH = os.path.join(CATALOG_DIR, 'catalog.parquet')
catalog = pd.read_parquet(CATALOG_PATH)
if 'rating_count' not in catalog.columns:                       # guard: safe to re-run
    _r = pd.read_csv(os.path.join(DATA, 'ratings.csv'), usecols=['movieId'], dtype={'movieId':'int32'})
    counts = _r['movieId'].value_counts()
    catalog['rating_count'] = catalog['movieId'].map(counts).fillna(0).astype('int32')
    catalog.to_parquet(CATALOG_PATH, index=False)
print(catalog.sort_values('rating_count', ascending=False)[['title','rating_count']].head(5).to_string(index=False))

                   title  rating_count
The Shawshank Redemption        102929
            Forrest Gump        100296
            Pulp Fiction         98409
              The Matrix         93808
The Silence of the Lambs         90330


write the re-ranker module

In [ ]:
%%writefile /content/drive/MyDrive/Projects/multi-agent-discovery/src/tools/reranker.py
"""Agent-invokable popularity re-ranker. Blends a popularity prior into semantic candidates.
Kept separate from Tool B so the PLANNER decides when to apply it (popular vs hidden-gems)."""
import numpy as np
import pandas as pd


class PopularityReranker:
    def __init__(self, catalog_path):
        pop = pd.read_parquet(catalog_path).set_index("movieId")["rating_count"]
        self.pop = pop.to_dict()
        self._max_log = float(np.log1p(max(self.pop.values()))) if self.pop else 1.0

    def _prior(self, movie_id):                       # 0..1 popularity, log-scaled
        return float(np.log1p(self.pop.get(int(movie_id), 0)) / self._max_log)

    def rerank(self, results, pop_weight=0.15, n=None):
        ranked = []
        for r in results:                              # results = Tool B output (movieId + cosine 'score')
            prior = self._prior(r["movieId"])
            ranked.append({**r,
                           "popularity_prior": round(prior, 3),
                           "blended_score": round(r["score"] + pop_weight * prior, 4)})
        ranked.sort(key=lambda x: x["blended_score"], reverse=True)
        return ranked[:n] if n else ranked

Writing /content/drive/MyDrive/Projects/multi-agent-discovery/src/tools/reranker.py


Verify against the rank-69 (Star Wars) finding



In [ ]:
import sys; sys.path.append(os.path.join(PROJECT_ROOT, 'src'))
sys.modules.pop('tools.tool_b_semantic', None); sys.modules.pop('tools.reranker', None)
from tools.tool_b_semantic import ToolB
from tools.reranker import PopularityReranker

tool_b   = ToolB(FAISS_DIR, CATALOG_PATH, device='cuda' if torch.cuda.is_available() else 'cpu')
reranker = PopularityReranker(CATALOG_PATH)
title_of = catalog.set_index('movieId')['title']
SW = 260   # Star Wars

res = tool_b.semantic_search("a rebellion fights an evil galactic empire in space", n=200)["results"]
raw_ids = [r["movieId"] for r in res]
print("SW raw semantic rank:", (raw_ids.index(SW)+1) if SW in raw_ids else ">200")

# pop ON (favor popular)
on = reranker.rerank(res, pop_weight=0.15, n=10); on_ids=[r["movieId"] for r in on]
print("SW rank with pop_weight=0.15:", (on_ids.index(SW)+1) if SW in on_ids else ">10")
for r in on[:5]:
    print(f"   cos={r['score']:.3f} pop={r['popularity_prior']:.2f} -> {r['blended_score']:.3f}  {title_of.get(r['movieId'])}")

# pop OFF (hidden gems) -> should leave SW where it was
off = reranker.rerank(res, pop_weight=0.0, n=10); off_ids=[r["movieId"] for r in off]
print("SW rank with pop_weight=0.0:", (off_ids.index(SW)+1) if SW in off_ids else ">10")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

SW raw semantic rank: 75
SW rank with pop_weight=0.15: >10
   cos=0.582 pop=0.96 -> 0.726  Return of the Jedi
   cos=0.574 pop=0.88 -> 0.706  Star Wars: Episode III - Revenge of the Sith
   cos=0.581 pop=0.82 -> 0.703  Rogue One: A Star Wars Story
   cos=0.588 pop=0.65 -> 0.685  Wing Commander
   cos=0.549 pop=0.85 -> 0.677  Star Wars: The Force Awakens
SW rank with pop_weight=0.0: >10
